此notebook 面向官方 Track 1 测试格式设计：`test/track1.jsonl` 只包含 `Consistent Value Response`。

核心方案：

- 训练、验证、测试都只使用 `Consistent Value Response`
- Prompt 中加入 README 的 19 个官方价值观定义
- 训练时只对答案标签 token 计算 loss
- 推理时不自由生成，而是对 19 个候选标签计算 conditional log-likelihood，选择最高分
- 官方测试集出现后，自动生成 `track1.pred.jsonl`，每行格式为 `{"Value": "..."}`

In [1]:
import os

os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
os.environ["HF_HUB_ETAG_TIMEOUT"] = "60"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "600"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

print("HF_ENDPOINT:", os.environ.get("HF_ENDPOINT"))
print("HF_HUB_DISABLE_XET:", os.environ.get("HF_HUB_DISABLE_XET"))

HF_ENDPOINT: https://hf-mirror.com
HF_HUB_DISABLE_XET: 1


In [ ]:
MODEL_NAME    = "unsloth/Qwen3-4B"             # 可替换为其他模型
DATA_DIR = "/home/yangdejin/nlpcc/nlpcc_task2/data"
OUTPUT_DIR = "/home/yangdejin/nlpcc/nlpcc_task2/outputs/decoder/qwen3/E0"
MAX_SEQ_LEN = 1024
BATCH_SIZE = 8
GRAD_ACCUM = 4
NUM_EPOCHS = 4
LR = 1e-5
WARMUP_RATIO = 0.1
WEIGHT_DECAY = 0.01

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LOAD_IN_4BIT = True
SEED = 3407

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Model : {MODEL_NAME}")
print(f"Data  : {DATA_DIR}")
print(f"Output: {OUTPUT_DIR}")

Model : unsloth/Qwen3-4B
Data  : /home/yangdejin/nlpcc/nlpcc_task2/data
Output: /home/yangdejin/nlpcc/nlpcc_task2/outputs/decoder/qwen3


In [6]:
# Cell 3: 标签与官方定义
VALUE_DEFINITIONS = [
    ("Self-direction–thought", "Freedom to cultivate one's own ideas and abilities"),
    ("Self-direction–action", "Freedom to determine one's own actions"),
    ("Stimulation", "Excitement, novelty, and change"),
    ("Hedonism", "Pleasure and sensuous gratification"),
    ("Achievement", "Success according to social standards"),
    ("Power–dominance", "Power through exercising control over people"),
    ("Power–resources", "Power through control of material and social resources"),
    ("Face", "Security and power through maintaining one's public image and avoiding humiliation"),
    ("Security–personal", "Safety in one's immediate environment"),
    ("Security–societal", "Safety and stability in the wider society"),
    ("Tradition", "Maintaining and preserving cultural, family, or religious traditions"),
    ("Conformity–rules", "Compliance with rules, laws, and formal obligations"),
    ("Conformity–interpersonal", "Avoidance of upsetting or harming other people"),
    ("Humility", "Recognizing one's insignificance in the larger scheme of things"),
    ("Benevolence–dependability", "Being a reliable and trustworthy member of the ingroup"),
    ("Benevolence–caring", "Devotion to the welfare of ingroup members"),
    ("Universalism–concern", "Commitment to equality, justice, and protection for all people"),
    ("Universalism–nature", "Preservation of the natural environment"),
    ("Universalism–tolerance", "Acceptance and understanding of those who are different from oneself"),
]

VALUE_LABELS = [label for label, _ in VALUE_DEFINITIONS]
LABEL2ID = {label: i for i, label in enumerate(VALUE_LABELS)}
ID2LABEL = {i: label for label, i in LABEL2ID.items()}

print(f"Number of classes: {len(VALUE_LABELS)}")


Number of classes: 19


In [7]:
# Cell 4: 数据读取与 prompt 构造
import json
from pathlib import Path
from collections import Counter

PROMPT_HEADER = """You are a human-values classifier.
Choose exactly one Schwartz basic human value for the response.
Return only the value label, with no explanation.

Value labels:
{definitions}

Response:
{response}

### Answer:
"""

def read_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def get_response(row):
    return str(row.get("Consistent Value Response", "") or "").strip()

def get_label(row):
    label = row.get("Value")
    if label not in LABEL2ID:
        raise ValueError(f"Unknown label: {label!r}")
    return str(label)

def build_prompt(response):
    definitions = "\n".join(f"- {label}: {desc}" for label, desc in VALUE_DEFINITIONS)
    return PROMPT_HEADER.format(definitions=definitions, response=response.strip())

train_rows = read_jsonl(os.path.join(DATA_DIR, "train.jsonl"))
dev_rows = read_jsonl(os.path.join(DATA_DIR, "dev.jsonl"))

print(f"Train rows: {len(train_rows)}")
print(f"Dev rows  : {len(dev_rows)}")
print("Input fields used: Consistent Value Response only")

counter = Counter(get_label(row) for row in train_rows)
for label in VALUE_LABELS:
    print(f"{label:<36s} {counter[label]}")

print("\nPrompt preview:")
print(build_prompt(get_response(train_rows[0]))[:1200])


Train rows: 3520
Dev rows  : 514
Input fields used: Consistent Value Response only
Self-direction–thought               119
Self-direction–action                124
Stimulation                          400
Hedonism                             164
Achievement                          174
Power–dominance                      156
Power–resources                      237
Face                                 258
Security–personal                    202
Security–societal                    70
Tradition                            90
Conformity–rules                     385
Conformity–interpersonal             236
Humility                             100
Benevolence–dependability            189
Benevolence–caring                   317
Universalism–concern                 160
Universalism–nature                  71
Universalism–tolerance               68

Prompt preview:
You are a human-values classifier.
Choose exactly one Schwartz basic human value for the response.
Return only the value labe

In [8]:
# Cell 5: 加载模型与 LoRA
import random
import numpy as np
import torch
from unsloth import FastLanguageModel
from transformers import set_seed

set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,
    load_in_4bit=LOAD_IN_4BIT,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable:,} / {total:,} ({100 * trainable / total:.2f}%)")


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.4: Fast Qwen3 patching. Transformers: 4.57.6. vLLM: 0.19.1.
   \\   /|    NVIDIA GeForce RTX 3090. Num GPUs = 1. Max memory: 23.595 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
unsloth/qwen3-4b-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.5.4 patched 36 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


Trainable parameters: 33,030,144 / 2,541,616,640 (1.30%)


In [9]:
# Cell 6: Dataset 与 response-only loss collator
from dataclasses import dataclass
from torch.utils.data import Dataset
from torch.nn.utils.rnn import pad_sequence

class ResponseOnlyDataset(Dataset):
    def __init__(self, rows, tokenizer, max_seq_len):
        self.features = []
        eos = tokenizer.eos_token or ""

        for row in rows:
            prompt = build_prompt(get_response(row))
            label = get_label(row)
            prompt_ids = tokenizer(prompt, add_special_tokens=False).input_ids
            label_ids = tokenizer(label + eos, add_special_tokens=False).input_ids

            if len(prompt_ids) + len(label_ids) > max_seq_len:
                keep = max_seq_len - len(label_ids)
                if keep <= 0:
                    raise ValueError("MAX_SEQ_LEN is too small to fit a label.")
                prompt_ids = prompt_ids[:keep]

            input_ids = prompt_ids + label_ids
            labels = [-100] * len(prompt_ids) + label_ids
            attention_mask = [1] * len(input_ids)

            self.features.append({
                "input_ids": torch.tensor(input_ids, dtype=torch.long),
                "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
                "labels": torch.tensor(labels, dtype=torch.long),
            })

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        return self.features[idx]

@dataclass
class CausalLMCollator:
    pad_token_id: int

    def __call__(self, features):
        input_ids = pad_sequence(
            [f["input_ids"] for f in features],
            batch_first=True,
            padding_value=self.pad_token_id,
        )
        attention_mask = pad_sequence(
            [f["attention_mask"] for f in features],
            batch_first=True,
            padding_value=0,
        )
        labels = pad_sequence(
            [f["labels"] for f in features],
            batch_first=True,
            padding_value=-100,
        )
        return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}

train_dataset = ResponseOnlyDataset(train_rows, tokenizer, MAX_SEQ_LEN)
data_collator = CausalLMCollator(pad_token_id=tokenizer.pad_token_id)

sample = train_dataset[0]
print({k: tuple(v.shape) for k, v in sample.items()})
print("Label tokens with loss:", int((sample["labels"] != -100).sum()))


{'input_ids': (361,), 'attention_mask': (361,), 'labels': (361,)}
Label tokens with loss: 7


In [10]:
# Cell 7: 训练
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LR,
    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,
    lr_scheduler_type="cosine",
    optim="adamw_8bit",
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=20,
    save_strategy="epoch",
    report_to="none",
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=data_collator,
    tokenizer=tokenizer,
)

trainer_stats = trainer.train()
print(trainer_stats)

adapter_dir = os.path.join(OUTPUT_DIR, "adapter")
model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
print(f"Saved adapter to {adapter_dir}")


/home/yangdejin/miniconda3/envs/nlpcc/lib/python3.12/site-packages/unsloth/models/_utils.py:2381: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  _original_trainer_init(self, *args, **kwargs)
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 3,520 | Num Epochs = 4 | Total steps = 440
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 4 x 1) = 32
 "-____-"     Trainable parameters = 33,030,144 of 4,055,498,240 (0.81% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
20,4.837800
40,4.124900
60,2.059100
80,0.406200
100,0.137300
120,0.079700
140,0.090900
160,0.080500
180,0.053600
200,0.075800


TrainOutput(global_step=440, training_loss=0.5692352682352066, metrics={'train_runtime': 3260.6149, 'train_samples_per_second': 4.318, 'train_steps_per_second': 0.135, 'total_flos': 1.1418368811405312e+17, 'train_loss': 0.5692352682352066, 'epoch': 4.0})
Saved adapter to /home/yangdejin/nlpcc/nlpcc_task2/outputs/decoder/qwen3/adapter


In [11]:
# Cell 8: 19 类 label likelihood 推理函数
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

def compute_candidate_scores(model, tokenizer, prompt, max_seq_len):
    eos = tokenizer.eos_token or ""
    prompt_ids = tokenizer(prompt, add_special_tokens=False).input_ids
    full_ids = []
    candidate_spans = []

    for label in VALUE_LABELS:
        label_ids = tokenizer(label + eos, add_special_tokens=False).input_ids
        ids = prompt_ids + label_ids
        if len(ids) > max_seq_len:
            keep = max_seq_len - len(label_ids)
            ids = prompt_ids[:keep] + label_ids
            span_start = keep
        else:
            span_start = len(prompt_ids)
        full_ids.append(torch.tensor(ids, dtype=torch.long))
        candidate_spans.append((span_start, len(ids)))

    device = next(model.parameters()).device
    batch = pad_sequence(full_ids, batch_first=True, padding_value=tokenizer.pad_token_id).to(device)
    attention_mask = (batch != tokenizer.pad_token_id).long()

    with torch.no_grad():
        logits = model(input_ids=batch, attention_mask=attention_mask).logits
        log_probs = torch.log_softmax(logits[:, :-1], dim=-1)

    scores = []
    for i, (span_start, span_end) in enumerate(candidate_spans):
        token_positions = torch.arange(span_start, span_end, device=device)
        pred_positions = token_positions - 1
        token_ids = batch[i, token_positions]
        token_log_probs = log_probs[i, pred_positions, token_ids]
        scores.append(float(token_log_probs.mean().detach().cpu()))
    return scores

def predict_rows(rows, desc="Scoring labels"):
    FastLanguageModel.for_inference(model)
    model.eval()
    preds = []
    for row in tqdm(rows, desc=desc):
        prompt = build_prompt(get_response(row))
        scores = compute_candidate_scores(model, tokenizer, prompt, MAX_SEQ_LEN)
        preds.append(VALUE_LABELS[int(np.argmax(scores))])
    return preds


In [12]:
# Cell 9: Dev 评估
dev_preds = predict_rows(dev_rows, desc="Dev")
dev_gold = [get_label(row) for row in dev_rows]

y_true = [LABEL2ID[x] for x in dev_gold]
y_pred = [LABEL2ID[x] for x in dev_preds]

acc = accuracy_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, average="macro")
print(f"Dev accuracy: {acc:.4f}")
print(f"Dev macro F1: {macro_f1:.4f}")
print(classification_report(y_true, y_pred, target_names=VALUE_LABELS, digits=4))

dev_pred_path = os.path.join(OUTPUT_DIR, "dev_predictions.jsonl")
with open(dev_pred_path, "w", encoding="utf-8") as f:
    for row, pred in zip(dev_rows, dev_preds):
        f.write(json.dumps({
            "gold": get_label(row),
            "pred": pred,
            "response": get_response(row),
        }, ensure_ascii=False) + "\n")
print(f"Saved dev predictions to {dev_pred_path}")


Dev:   0%|          | 0/514 [00:00<?, ?it/s]

Dev accuracy: 0.8813
Dev macro F1: 0.8576
                           precision    recall  f1-score   support

   Self-direction–thought     0.7222    0.7647    0.7429        17
    Self-direction–action     0.6471    0.6111    0.6286        18
              Stimulation     1.0000    1.0000    1.0000        58
                 Hedonism     1.0000    1.0000    1.0000        24
              Achievement     0.9167    0.8462    0.8800        26
          Power–dominance     0.9200    1.0000    0.9583        23
          Power–resources     0.9714    0.9714    0.9714        35
                     Face     0.9118    0.8378    0.8732        37
        Security–personal     0.8235    0.9655    0.8889        29
        Security–societal     1.0000    0.8182    0.9000        11
                Tradition     0.9333    1.0000    0.9655        14
         Conformity–rules     0.9643    0.9818    0.9730        55
 Conformity–interpersonal     0.7500    0.7941    0.7714        34
                 Hu